In [ ]:
!pip install bertopic pyBigKinds

In [ ]:
import os
os.chdir("..")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import pyBigKinds as pbk
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

In [ ]:
def list_to_str(words: list):
    """Function that list data change to string for text preprocessing"""
    for i in range(len(words)):
        text = ""
        for word in words[i]:
            if text == "":
                text = word
            else:
                text = text + " " + word
        words[i] = text
    return words

In [ ]:
all_df = pd.read_excel("./data/bipolar_disorder_all.xlsx", engine="openpyxl")

conditions = [
    (all_df["언론사"]==all_df["언론사"].unique()[0]),
    (all_df["언론사"]==all_df["언론사"].unique()[1]),
    (all_df["언론사"]==all_df["언론사"].unique()[2]),
    (all_df["언론사"]==all_df["언론사"].unique()[3]),
    (all_df["언론사"]==all_df["언론사"].unique()[4]),
    (all_df["언론사"]==all_df["언론사"].unique()[5]),
    (all_df["언론사"]==all_df["언론사"].unique()[6]),
    (all_df["언론사"]==all_df["언론사"].unique()[7]),
    (all_df["언론사"]==all_df["언론사"].unique()[8]),
    (all_df["언론사"]==all_df["언론사"].unique()[9]),
    (all_df["언론사"]==all_df["언론사"].unique()[10]),
    (all_df["언론사"]==all_df["언론사"].unique()[11]),
    (all_df["언론사"]==all_df["언론사"].unique()[12]),
    (all_df["언론사"]==all_df["언론사"].unique()[13]),
]
choices = ["동아일보", "조선일보", "경향신문", "조선일보", "한겨레", "중앙일보",
           "서울신문", "세계일보", "한겨레", "국민일보", "문화일보", "한국일보", "경향신문", "동아일보"]

all_df["언론사"] = np.select(conditions, choices)

In [ ]:
df = all_df[all_df["키워드"].isna() == False]
df = df[df["언론사"].isin(["조선일보", "동아일보", "한겨레", "경향신문", "한국일보", "서울신문"])]

print(all_df.shape)
print(df.shape)

In [ ]:
df["연도"] = df["일자"].dt.year
df = df.sort_values("연도")
df.reset_index(drop=True, inplace=True)

words = pbk.keyword_parser(pbk.keyword_list(df))
words = list_to_str(words)
timestamp = df["연도"].tolist()

In [ ]:
vectorizer_model = CountVectorizer(ngram_range=(1, 2))
vectorizer_model.fit(words)
vectorizer_model.vocabulary_

In [ ]:
# tunning the BERTopic model
sentence_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_samples=20, gen_min_span_tree=True, prediction_data=False, min_cluster_size=10)

# defind BERTopic Model
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    embedding_model=sentence_model,
    ctfidf_model=ctfidf_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    top_n_words = 20,
    verbose=True
)

In [ ]:
# fit the data
topics, probs = topic_model.fit_transform(words)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_info = topic_model.get_topic_info()
topic_words = [topic_model.get_topic(topic) for topic in topic_info.Topic]

result_df = pd.DataFrame({
        'Topic': topic_info['Topic'],
        'Count': topic_info['Count'],
        'Name': topic_info['Name'],
        'Top Words': [', '.join([word for word, _ in words]) for words in topic_words]
    })
result_df.to_excel('bipolar_disorder/bertopic_bipolar.xlsx', index=False)

In [ ]:
df_docs = pd.DataFrame({
    'name': df['제목'],
    'Year': df['일자'],
    'Document': df['키워드'],
    'Topic': topics
    })
df_docs.to_excel('bipolar_disorder/doc_topics_bipolar.xlsx', index=False)

In [ ]:
df.to_excel('bipolar_disorder/df_bipolar.xlsx', index=False)

In [ ]:
topic_model.save("./bin/bipolar_model.bin")